In [1]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import GWFish.modules as gw
import gc
import torch
from dingo.core.posterior_models.build_model import build_model_from_kwargs
import pandas as pd
from dingo.gw.inference.sampler import GWComposedSampler
import dingo.gw.injection as injection
from dingo.gw.noise.asd_dataset import ASDDataset
import shutil



os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # use GPU 1
os.environ["taskset -c"] = "0-48"  # use CPU cores 0-48
waveform_model = 'IMRPhenomXPHM'
f_ref = 20.
detectors = ['ET']
network = gw.detection.Network(detector_ids = detectors, detection_SNR = (0., 8.))


/home/tpausch/.conda/envs/ET-DINGO/lib/python3.10/site-packages/lalsimulation/_lalsimulation_swig.py:8: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [6]:
import glob, os, pickle
import numpy as np

G, c, Msun = 6.674e-11, 2.998e8, 1.989e30

def scan_detector(d_f, freqs, psd, Mc_grid, fmin, fmax, frac=0.5, min_bins=20):
    N = 2*(len(freqs)-1); df = freqs[1]-freqs[0]
    out = np.full(len(Mc_grid), np.nan)
    for i, Mc in enumerate(Mc_grid):
        f_isco = 4400/(2.3*Mc)                      # equal-mass estimate
        hi = min(fmax, frac*f_isco)
        m = (freqs >= fmin) & (freqs <= hi) & (psd > 0)
        if m.sum() < min_bins:
            continue
        idx = np.where(m)[0]; f = freqs[m]
        psi = (3/128) * (np.pi*G*Mc*Msun*f/c**3)**(-5/3)
        h = f**(-7/6) * np.exp(1j*psi)
        norm2 = 4*df*np.sum(np.abs(h)**2/psd[m])
        z = np.zeros(N, dtype=complex)
        z[idx] = d_f[m]*np.conj(h)/psd[m]
        out[i] = np.max(np.abs(np.fft.ifft(z)*N))**2 / norm2
    return out

def estimate_Mc(strain_data, delta_f, Mc_grid, fmin, fmax):
    data, asds = strain_data["waveform"], strain_data["asds"]
    total = np.zeros(len(Mc_grid))
    for det in data:
        d = np.asarray(data[det])
        freqs = np.arange(len(d))*delta_f
        s = scan_detector(d, freqs, np.asarray(asds[det])**2, Mc_grid, fmin, fmax)
        total += np.nan_to_num(s, nan=0.0)
    if not np.any(total > 0):
        return np.nan, total
    return Mc_grid[np.argmax(total)], total

inj_dir = "/scratch/tpausch/test_data/stage1/injections"
files = sorted(glob.glob(f"{inj_dir}/inj_*.pkl"))
print(len(files), "files")

# inspect one file before running everything
p0 = pickle.load(open(files[0], "rb"))
print(p0.keys())
print(p0["strain_data"].keys())
for k, v in p0["strain_data"]["waveform"].items():
    print(k, np.shape(v), np.asarray(v).dtype)
print(p0["theta_true"]["chirp_mass"])

1000 files
dict_keys(['theta_true', 'strain_data'])
dict_keys(['parameters', 'waveform', 'asds'])
ET-1 (2049,) complex128
ET-2 (2049,) complex128
ET-3 (2049,) complex128
911.7553062171859


In [11]:
Mc_grid = np.geomspace(50, 1000, 1000)   # match prior["chirp_mass"] range
delta_f = 1/8                             # replace with the value from metadata

rows = []
for fn in files[:100]:
    p = pickle.load(open(fn, "rb"))
    Mc_hat, _ = estimate_Mc(p["strain_data"], delta_f, Mc_grid, fmin=2, fmax=20)
    idx = int(os.path.basename(fn)[4:8])
    rows.append((idx, p["theta_true"]["chirp_mass"], Mc_hat))

idx, true, est = np.array(rows).T
ok = np.isfinite(est)
rel_err = (est[ok]-true[ok])/true[ok]
print(f"{(~ok).sum()} without usable band, median rel err {np.median(rel_err):.3f}, std {np.std(rel_err):.3f}")

0 without usable band, median rel err -0.841, std 0.099


In [12]:
print("true Mc percentiles:", np.percentile(true, [0, 16, 50, 84, 100]))
print("est  Mc percentiles:", np.percentile(est,  [0, 16, 50, 84, 100]))
print("fraction at grid min:", np.mean(est <= Mc_grid[0]*1.001))
print("fraction of injections with true Mc < 150:", np.mean(true < 150))

true Mc percentiles: [144.39342038 455.74627449 748.80446137 911.1359945
 997.98980316]
est  Mc percentiles: [ 54.21668718  98.38805202 119.30163056 127.43763133
 141.54004265]
fraction at grid min: 0.0
fraction of injections with true Mc < 150: 0.01


In [5]:
import glob, os
base = "/scratch/tpausch/test_data/stage1"
print(os.path.exists(base))
print(os.listdir(base))
for sub in os.listdir(base):
    print(sub, glob.glob(f"{base}/{sub}/*")[:5])

True
['results', 'injections']
results ['/scratch/tpausch/test_data/stage1/results/result_0857.pkl', '/scratch/tpausch/test_data/stage1/results/result_0425.pkl', '/scratch/tpausch/test_data/stage1/results/result_0382.pkl', '/scratch/tpausch/test_data/stage1/results/result_0044.pkl', '/scratch/tpausch/test_data/stage1/results/result_0095.pkl']
injections ['/scratch/tpausch/test_data/stage1/injections/inj_0848.pkl', '/scratch/tpausch/test_data/stage1/injections/inj_0899.pkl', '/scratch/tpausch/test_data/stage1/injections/inj_0214.pkl', '/scratch/tpausch/test_data/stage1/injections/inj_0675.pkl', '/scratch/tpausch/test_data/stage1/injections/inj_0103.pkl']
